In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact
import info
import loader
import stats
import plotter

dataset_path = "../dataset/processed/features.csv"
    
try:
  df = loader.load_data(dataset_path)
except FileNotFoundError as e:
  print(f"Error: dataset file ({dataset_path}) not found")
  sys.exit(1)

### Numero di famiglie linguistiche, sottofamiglie, genus e lingue all'interno del dataset

In [ ]:
categories_count = df[["Family", "Subfamily", "Genus", "Language_ID"]].nunique()
plotter.bar_plot(
  categories_count, 
  title="Diversità linguistica del dataset",
  xlabel="Livelli tassonomici",
  ylabel="Numero di categorie uniche",
  annotate=True
)

### Contare il numero di lingue per ciascuna macroarea nel dataset

In [ ]:
qdf = stats.count(df, ["Language_ID"], "Macroarea", sort=True)

plotter.bar_plot(
  qdf,
  title="Numero di lingue per macroarea",
  xlabel="Macroaree",
  ylabel="Numero di lingue",
  annotate=True
)

### Contare il numero di lingue per Paese nel dataset

In [ ]:
qdf = stats.count(df, ["Language_ID"], "Country_Name", sort=True).squeeze()

n_top = 120
languages_by_family = stats.pack_from(qdf, n_top).sort_values(ascending=False)
languages_by_family_chunks = stats.get_chunks(languages_by_family, n_top // 2)

for chunk in languages_by_family_chunks:
  plotter.bar_plot(
    chunk,
    figsize=(30, 10),
    title="Numero di lingue per Paese",
    xlabel="Paese",
    ylabel="Numero di lingue",
    annotate=False
  )


### contare il numero di lingue per famiglia linguistica nel dataset

In [ ]:
n_top= 120
qdf = stats.count(df, ["Language_ID"], "Family", sort=True).squeeze()

# Questo perché era già presente un campo other nel dataset
previous_other_count = qdf.loc["other"]
qdf = qdf.drop(index="other")

languages_by_family = stats.pack_from(qdf, n_top)
languages_by_family.at["Other"] += previous_other_count

# Una volta sommato il valore precedente di other bisogna riordinare
languages_by_family = languages_by_family.sort_values(ascending=False)

languages_by_family_chunks = stats.get_chunks(languages_by_family, n_top // 2)

for chunk in languages_by_family_chunks:
  plotter.bar_plot(
    chunk,
    figsize=(30, 10),
    title="Numero di lingue per famiglia linguistica",
    xlabel="Famiglie",
    ylabel="Numero di lingue",
    annotate=False
)

n_top = 30
languages_by_family = stats.pack_from(qdf, n_top).sort_values(ascending=False)

plotter.pie_plot(
    languages_by_family,
    figsize=(10, 10),
    title="Distribuzione delle lingue per famiglia linguistica"
)

### contare il numero di lingue per genus nel dataset

In [ ]:
n_top= 120
qdf = stats.count(df, ["Language_ID"], "Genus", sort=True).squeeze()

languages_by_genus = stats.pack_from(qdf, n_top).sort_values(ascending=False)
languages_by_genus_chunks = stats.get_chunks(languages_by_genus, n_top // 2)

for chunk in languages_by_genus_chunks:
  plotter.bar_plot(
    chunk,
    figsize=(30, 10),
    title="Numero di lingue per genus",
    xlabel="Genus",
    ylabel="Numero di lingue",
    annotate=False
)

n_top = 30
languages_by_genus = stats.pack_from(qdf, n_top).sort_values(ascending=False)

plotter.pie_plot(
    languages_by_genus,
    figsize=(10, 10),
    title="Distribuzione delle lingue per famiglia linguistica"
)

### Conteggio di famiglie linguistiche e genus per macroarea

In [ ]:
qdf = stats.count(df, ["Family"], "Macroarea", sort=True)

plotter.bar_plot(
  qdf,
  title="Numero di macro famiglie per macroarea",
  annotate=True
)

qdf = stats.count(df, ["Genus"], "Macroarea", sort=True)

plotter.bar_plot(
  qdf,
  title="Numero di genus per macroarea",
  annotate=True
)

### Conteggio di lingue divise in famiglie per Paese

In [ ]:
qdf = df.drop_duplicates(subset=["Country_Name", "Family", "Language_ID"])
qdf = pd.crosstab(qdf["Country_Name"], qdf["Family"])

qdf = qdf.assign(Total=qdf.sum(axis=1))

languages_by_family = qdf.sort_values("Total", ascending=False)
languages_by_family = languages_by_family.drop("Total", axis=1)
first_chunk = stats.get_chunks(languages_by_family, 40)[0]

plotter.stacked_bar_plot(first_chunk, legend=False, title="Lingue per famiglia suddivise per Paese")

### Conteggio di lingue divise in genus per Paese

Ad occhio si possono vedere quali Paesi sono più linguisticamente vari rispetto ad altri.

In [ ]:
qdf = df.drop_duplicates(subset=["Country_Name", "Genus", "Language_ID"])
qdf = pd.crosstab(qdf["Country_Name"], qdf["Genus"])

qdf = qdf.assign(Total=qdf.sum(axis=1))

languages_by_family = qdf.sort_values("Total", ascending=False)
languages_by_family = languages_by_family.drop("Total", axis=1)
first_chunk = stats.get_chunks(languages_by_family, 40)[0]

plotter.stacked_bar_plot(first_chunk, legend=False, title="Lingue per genus suddivise per Paese")

### Distribuzione delle lingue in famiglie dato un certo Paese

In [ ]:
def select_country(countries):
  country_name = countries
  qdf = df.copy()
  qdf = stats.filter(qdf, "Country_Name", [country_name])
  qdf = qdf.drop_duplicates(subset=["Family", "Language_ID"])
  qdf = stats.count(qdf, ["Language_ID"], "Family", sort=True).squeeze()

  plotter.bar_plot(
    qdf,
    figsize=(10, 10),
    title=f"Numero di lingue per famiglia in {country_name}",
    annotate=True
  )

  plotter.pie_plot(
    qdf,
    figsize=(10, 10),
    title=f"Numero di lingue per famiglia in {country_name}"
  )

interact(select_country, countries=stats.get_list_of(df, "Country_Name"));


### Distribuzione delle lingue in genus dato un certo Paese

In [ ]:
def select_country(countries):
  country_name = countries
  qdf = df.copy()
  qdf = stats.filter(qdf, "Country_Name", [country_name])
  qdf = qdf.drop_duplicates(subset=["Genus", "Language_ID"])
  qdf = stats.count(qdf, ["Language_ID"], "Genus", sort=True).squeeze()

  plotter.bar_plot(
    qdf,
    figsize=(10, 10),
    title=f"Numero di lingue per Genus in {country_name}"
  )

  plotter.pie_plot(
    qdf,
    figsize=(20, 10),
    title=f"Numero di lingue per Genus in {country_name}"
  )

interact(select_country, countries=stats.get_list_of(df, "Country_Name"));

### Disposizione dei paesi in base al numero di famiglie e numero di lingue

Un Paese può avere un alto numero di lingue, ma ciò non lo rende più linguisticamente vario rispetto a un Paese con un numero inferiore di lingue, ma suddivisie in un maggior numero di famiglie linguistiche.

In [ ]:
qdf = stats.count(df, ["Family", "Language_ID"], "Country_Name", sort=True).head(20)

plotter.scatter_plot(qdf, figsize=(10, 10), x="Family", y="Language_ID", annotate=True, title="Diversità linguistica per Paese in termini di famiglie")


In [ ]:
qdf = stats.count(df, ["Genus", "Language_ID"], "Country_Name", sort=True).head(20)

plotter.scatter_plot(qdf, figsize=(10, 10), x="Genus", y="Language_ID", annotate=True, title="Diversità linguistica per Paese in termini di Genus")